# 02 — Nhập riêng data mới 00008 (Colab CPU)

**CPU → Run all trên một tài khoản riêng. Không dừng luồng submit.**
Chỉ nhận `vibiomir_retain_shard_00008.tar` mới tìm thấy trong AI Guru.
Tự kiểm descriptor, snapshot và SHA frontier qua Drive API ranges trước khi khóa plan.
Không tải cả archive 2,50 GB; metadata cần xử lý và kết quả được lưu trên Drive.

Dùng lại mapping hợp lệ của baseline và 9 nguồn đã hoàn tất để bỏ qua phần trùng.
Chín nguồn cũ chỉ đọc receipt nhỏ khi nhận handoff; SHA của mapping được kiểm
khi thực sự dùng để chọn ID mới. Không chạy lại import/chunk của chúng.

Output: `VietMedBridge/data_staging/btc-new-71be0e12-00008-20261020-v1/`.
Không đổi active candidate, snapshot hoặc model cache của luồng submit.
Ngắt phiên: mở lại đúng notebook, Run all với cùng folder; giữ plan và checkpoint.
`INTAKE_COMPLETE` chỉ là import/quality/chunk CPU cho nguồn này, chưa phải embedding,
index, submission hoặc chứng nhận đã đủ toàn bộ ID của BTC.

In [ ]:
#@title 1. CPU: dùng code/runtime lock của campaign đã xong; output riêng cho 00008
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Dùng runtime Colab Python 3.11–3.13.")
print("Bắt đầu CPU intake; không nạp model embedding.", flush=True)
from google.colab import drive
drive.mount("/content/drive")

# Chỉ sửa nếu project của team ở vị trí khác; tự tìm shortcut dùng chung.
PROJECT = Path("/content/drive/MyDrive/VietMedBridge")
if not (PROJECT / "data").is_dir():
    shared = Path("/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge")
    if (shared / "data").is_dir():
        PROJECT = shared
    else:
        raise FileNotFoundError("Chưa tìm thấy folder VietMedBridge dùng chung. Thêm shortcut VietMedBridge vào My Drive.")
PARENT_ROOT = PROJECT.resolve() / "data_staging/btc-new-71be0e12-20261019-v1"
CAMPAIGN = "btc-new-71be0e12-00008-20261020-v1"
INTAKE_ROOT = PROJECT.resolve() / "data_staging" / CAMPAIGN
INTAKE_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/vmb_intake_00008_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKOUT = Path("/content/VietMedBridge-intake-00008")
REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
lock_path = INTAKE_ROOT / "intake_code_lock.json"
parent_lock = PARENT_ROOT / "intake_code_lock.json"
if not parent_lock.is_file():
    raise RuntimeError("Thiếu code lock của campaign 9 nguồn đã hoàn tất; chưa tự chọn code khác.")
lock = json.loads((lock_path if lock_path.is_file() else parent_lock).read_text())
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("Intake code lock belongs to another repository.")
if not (CHECKOUT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
if remote != REPO_URL:
    raise ValueError("Checkout intake thuộc repo khác.")
reference = lock.get("commit") or "main"
print("Tải code intake:", reference, flush=True)
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
commit = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True).strip()
if "vietmedbridge" in sys.modules and globals().get("_INTAKE_COMMIT") != commit:
    raise RuntimeError("Restart session để nạp đúng code intake; checkpoint trên Drive vẫn giữ.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", commit], check=True)
print("Cài dependencies CPU…", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CHECKOUT)], check=True)
runtime_path = INTAKE_ROOT / "intake_runtime_lock.json"
pinned_runtime_path = runtime_path if runtime_path.is_file() else PARENT_ROOT / "intake_runtime_lock.json"
if not pinned_runtime_path.is_file():
    raise RuntimeError("Thiếu runtime lock của parent campaign.")
if pinned_runtime_path.is_file():
    pinned = json.loads(pinned_runtime_path.read_text())
    packages = [name+"=="+version for name, version in pinned.items() if name != "vietmedbridge"]
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
sys.path.insert(0, str(CHECKOUT / "src"))
importlib.invalidate_caches()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/hf_cache_intake"
from vietmedbridge.artifacts import atomic_json, read_json, runtime_versions
from vietmedbridge.isolated_intake import intake_root
if INTAKE_ROOT != intake_root(PROJECT / "data", CAMPAIGN):
    raise ValueError("Intake output root không tách biệt khỏi submission.")
if not lock_path.is_file():
    atomic_json(lock_path, {"repo_url": REPO_URL, "commit": commit})
if not runtime_path.is_file():
    atomic_json(runtime_path, runtime_versions())
_INTAKE_COMMIT = commit
print("Code:", commit, "\nCheckpoint:", INTAKE_ROOT, flush=True)


In [ ]:
#@title 2. Nhận 9 receipt cũ và audit nguồn 00008
"""Manifest-first intake controller, embedded into the Colab notebook.

Kept outside src/ deliberately: changing resume orchestration must not change
the pinned producer fingerprint or invalidate existing document/chunk shards.
Completed receipts are NOT a fresh payload-integrity audit: only metadata is
read for status; mapping payloads are SHA-checked when actually used.
"""
from collections import Counter
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path
import re
import tempfile

from vietmedbridge import isolated_intake as intake_core
from vietmedbridge.artifacts import (atomic_json, code_fingerprint, digest_json,
    read_json, runtime_versions, sha256_file, utc_now, verify_file)
from vietmedbridge.dataset import _connection


def intake_memory_mb():
    """At most 2 GiB and a quarter of available RAM; no reservation or GPU use."""
    available = []
    try:
        match = re.search(r"^MemAvailable:\s+(\d+) kB", Path("/proc/meminfo").read_text(), re.M)
        if match:
            available.append(int(match[1])*1024)
    except OSError:
        pass
    for limit, used in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            available.append(max(0, int(Path(limit).read_text())-int(Path(used).read_text())))
        except (OSError, ValueError):
            pass
    return min(2048, max(64, min(available)//(4*1024**2))) if available else 512


def bounded_intake_parquet(con, query, destination, work_dir):
    """Flush wide text every 2,048 rows, then atomically publish the full file."""
    with tempfile.TemporaryDirectory(prefix="intake-table-", dir=work_dir) as temporary:
        path = Path(temporary)/"table.parquet"
        con.sql(query).write_parquet(str(path), compression="zstd", row_group_size=2048)
        return intake_core.publish_file(path, destination)


@contextmanager
def bounded_intake_sql(memory_mb=None):
    """Scoped I/O adapter for the single intake controller, including old locks.

    The pinned producer files, policies and completed selections stay untouched.
    Only this module's selection/outcome SQL and Parquet writer are adapted;
    normal importer/chunk shards still use their original producer.
    """
    budget = intake_memory_mb() if memory_mb is None else memory_mb
    if type(budget) is not int or not 64 <= budget <= 2048:
        raise ValueError("Intake SQL budget must be 64..2048 MiB.")
    original_connection, original_parquet = intake_core._connection, intake_core._parquet

    def connect(temp_dir, memory_limit=None):
        con = _connection(temp_dir, memory_limit=memory_limit or f"{budget}MiB")
        try:
            con.execute("SET threads=1")
            con.execute("SET preserve_insertion_order=false")
        except BaseException:
            con.close()
            raise
        return con

    intake_core._connection, intake_core._parquet = connect, bounded_intake_parquet
    print(f"Intake SQL: RAM limit {budget} MiB | 1 thread | local spill | Parquet groups 2,048 rows", flush=True)
    try:
        yield
    finally:
        intake_core._connection, intake_core._parquet = original_connection, original_parquet


class IntakeResume:
    def __init__(self, plan, *, root, work_dir, tokenizer_spec, chunk_config, shard_size=2048,
                 prior_mappings=()):
        self.plan, self.root, self.work = plan, Path(root), Path(work_dir)
        self.spec, self.config, self.shard_size = tokenizer_spec, chunk_config, shard_size
        self.plan_sha = digest_json(plan)
        self.prior_mappings = [(Path(path), checksum) for path, checksum in prior_mappings]
        if [checksum for _, checksum in self.prior_mappings] != plan.get("prior_mapping_sha256", []):
            raise ValueError("Prior intake mappings differ from the pinned continuation plan.")
        if any((p/"active_data_candidate.json").exists() for p in (self.root, *self.root.parents)):
            raise ValueError("Intake must stay outside the active submission data root.")
        if any(not re.fullmatch(r"[a-zA-Z0-9_-]+", s["job_id"]) for s in plan["sources"]):
            raise ValueError("Invalid intake job ID.")
        intake_core.pin_plan(self.root, plan)
        self.work.mkdir(parents=True, exist_ok=True)
        self.contract = {"tokenizer": tokenizer_spec, "chunking": asdict(chunk_config),
            "shard_size": shard_size, "code_sha256": code_fingerprint(), "runtime": runtime_versions()}
        contract = self.root/"intake_contract.json"
        if contract.exists() and read_json(contract) != self.contract:
            raise ValueError("Pinned intake code/runtime/chunk policy changed; do not discard checkpoints.")
        if not contract.exists():
            atomic_json(contract, self.contract)
        self.refresh()

    def _file(self, root, item):
        path = intake_core._inside(root, item["path"])
        stat = path.stat()  # Missing/changed files fail; never silently reprocess a completed source.
        if stat.st_size <= 0 or ("bytes" in item and stat.st_size != item["bytes"]):
            raise ValueError(f"Checkpoint artifact size changed: {path}")
        return path

    def refresh(self):
        self.completed = []
        gap = False
        for source in self.plan["sources"]:
            target = self.root/"sources"/source["job_id"]
            receipt = target/"complete.json"
            if not receipt.exists():
                gap = True
                continue
            if gap:
                raise ValueError("Out-of-order complete receipt; preserve files and inspect source lineage.")
            saved = read_json(receipt)
            if (saved.get("state") != "COMPLETE" or saved["source"] != source
                    or saved["plan_sha256"] != self.plan_sha):
                raise ValueError("Completed source/plan binding changed.")
            for kind, name, sha_key in (("added_ids", "added_ids.parquet", "added_ids_sha256"),
                                       ("outcomes", "source_outcomes.parquet", "outcomes_sha256")):
                item = saved.get("files", {}).get(kind, {"path": name, "sha256": saved[sha_key]})
                if item["sha256"] != saved[sha_key]:
                    raise ValueError("Receipt payload binding changed.")
                self._file(target, item)
            self.completed.append(saved)

    def status(self):
        counts = Counter()
        for saved in self.completed:
            counts.update(saved["counts"])
        return {"state": "INTAKE_COMPLETE" if len(self.completed) == len(self.plan["sources"]) else "INTAKE_IN_PROGRESS",
            "plan_sha256": self.plan_sha, "completed_sources": len(self.completed),
            "total_sources": len(self.plan["sources"]), "counts": dict(counts),
            "builds": [{"build_run": s["build_run"], "snapshot_sha256": s["build_snapshot_sha256"],
                "snapshot_id": s["source"]["snapshot_id"]} for s in self.completed if s["build_run"]],
            "resume_policy": "MANIFEST_FIRST_MAPPING_SHA_CHECKED_ON_USE",
            "scope": "CPU import, quality and exact-span chunks only; not embedded/frozen/promoted or submitted",
            "submission_candidate_changed": False, "updated_at": utc_now()}

    def _selection(self, source, target, existing):
        marker = target/"selection/selection.json"
        if not marker.exists():
            return None
        saved = read_json(marker)
        metadata = read_json(target/"metadata/metadata.json")
        expected = {"metadata": metadata,
            "links_sha256": self.plan["snapshots"][source["snapshot_id"]]["sha256"],
            "existing": [{"sha256": checksum} for _, checksum in existing]}
        if (saved["identity"] != expected or metadata["source"] != source
                or metadata["files"]["frontier"]["sha256"] != source["frontier_sha256"]
                or sum(saved["counts"].values()) != saved["official_ids"]):
            raise ValueError("Saved selection binding changed; no automatic rebuild.")
        for item in saved["files"].values():
            self._file(target/"selection", item)
        print(f"REUSE_SELECTION {source['job_id']} | no archive/baseline rescan", flush=True)
        return saved, metadata

    def _validated_build(self, source, selection, build_dir):
        path = build_dir/"build.json"
        if not path.exists():
            return None
        build = read_json(path)
        if not build.get("selected_range_complete"):
            return None
        if build.get("state") != "DATA_VALIDATED" or not build.get("integrity", {}).get("passed"):
            raise ValueError("Complete build has not passed integrity; retain and inspect it.")
        config = read_json(build_dir/"config.json")
        queued = selection["counts"].get("QUEUED_FOR_PIPELINE", 0)
        files = {kind: {"name": Path(intake_core.MEMBERS[kind]).name,
            "sha256": selection["files"][kind]["sha256"],
            "bytes": self._file(self.root/"sources"/source["job_id"]/"selection", selection["files"][kind]).stat().st_size}
            for kind in intake_core.MEMBERS}
        if (config["signature"] != digest_json({k: v for k, v in config.items() if k != "signature"})
                or config["signature"] != build["signature"]
                or config["source_input_signature"] != digest_json(files)
                or config["external_snapshot_id"] != source.get("original_snapshot_id", source["snapshot_id"])
                or config["official_links_sha256"] != self.plan["snapshots"][source["snapshot_id"]]["sha256"]
                or config["tokenizer"] != self.spec or config["chunks"] != asdict(self.config)
                or config["code_sha256"] != self.contract["code_sha256"]
                or config["runtime"] != self.contract["runtime"] or config["shard_size"] != self.shard_size
                or build["requested_input_records"] != queued or build["counts"]["input_records"] != queued
                or build["snapshot_sha256"] != digest_json({"signature": build["signature"], "parts": build["parts"]})):
            raise ValueError("Validated build identity differs from the saved selection/policy.")
        for part in build["parts"]:
            for kind, item in part["files"].items():
                path = self._file(build_dir, item)
                if kind in ("documents", "failures"):
                    # Only these payloads are about to be used to finish the
                    # ID/outcome mapping. Child/parent chunks stay deferred to freeze.
                    verify_file(path, item["sha256"])
        print(f"REUSE_VALIDATED_BUILD {source['job_id']} | no re-chunk or full validation replay", flush=True)
        return build

    @bounded_intake_sql()
    def run(self, *, load_reader, load_links, load_baseline, load_tokenizer, load_metadata=None,
            select_metadata=None, source_completed=None, max_new_sources=None):
        if max_new_sources is not None and (type(max_new_sources) is not int or max_new_sources < 1):
            raise ValueError("max_new_sources must be a positive integer.")
        newly_completed = 0
        self.refresh()
        for saved in self.completed:
            print(f"SKIP_COMPLETED_SOURCE {saved['source']['job_id']} | receipt only; payload checked on use", flush=True)
        if len(self.completed) == len(self.plan["sources"]):
            result = self.status()
            atomic_json(self.root/"intake_status.json", result)
            return result
        # Paths/checksums suffice until a NEW selection really needs a mapping.
        existing = [(None, self.plan["baseline"]["sha256"])] + self.prior_mappings + [
            (self.root/"sources"/s["source"]["job_id"]/"added_ids.parquet", s["added_ids_sha256"])
            for s in self.completed]
        for source in self.plan["sources"][len(self.completed):]:
            job = source["job_id"]
            target, build_dir = self.root/"sources"/job, self.root/"processed"/job
            print(f"CONTINUE_SOURCE {job} | {len(self.completed)}/{len(self.plan['sources'])} sources complete", flush=True)
            cached = self._selection(source, target, existing)
            if cached is None:
                metadata = (load_metadata(source, target/"metadata") if load_metadata is not None
                            else intake_core.stage_metadata(load_reader(), source, target/"metadata"))
                local_maps = [load_baseline() if path is None else intake_core._local_mapping(path, checksum, self.work)
                              for path, checksum in existing]
                for path, (_, checksum) in zip(local_maps, existing):
                    verify_file(path, checksum)
                links = load_links(source["snapshot_id"])
                selection = (select_metadata(metadata, links, local_maps, target/"selection") if select_metadata is not None
                    else intake_core.select_delta(target/"metadata", metadata, links, local_maps, target/"selection", self.work))
            else:
                selection, metadata = cached
            build = None
            if selection["counts"].get("QUEUED_FOR_PIPELINE", 0):
                build = self._validated_build(source, selection, build_dir)
                if build is None:
                    markers = sorted((build_dir/"parts").glob("*.done.json"))
                    committed = [read_json(path) for path in markers]
                    print(f"RESUME_PARTS {job} | {len(committed)} committed shards, "
                          f"{sum(p['records'] for p in committed):,} IDs; verify current source only", flush=True)
                    staged = intake_core.restore_import_source(target/"selection", selection, self.work/"sources"/job)
                    for kind in intake_core.MEMBERS:
                        item = selection["files"][kind]
                        verify_file(staged/item["path"], item["sha256"])
                    tokenizer, actual_spec = load_tokenizer()
                    if actual_spec != self.spec:
                        raise ValueError("Loaded tokenizer differs from the pinned contract.")
                    build = intake_core.import_external_corpus(staged, load_links(source["snapshot_id"]),
                        self.root/"processed", tokenizer, self.spec, run_name=job, config=self.config,
                        work_dir=self.work, shard_size=self.shard_size)
                    if not build["selected_range_complete"] or not build["integrity"]["passed"]:
                        raise ValueError("Intake build is not complete and valid.")
                atomic_json(build_dir/"intake_provenance.json", {"source": source,
                    "original_metadata": metadata["files"], "plan_sha256": self.plan_sha,
                    "membership_snapshot_id": source["snapshot_id"],
                    "original_snapshot_id": source.get("original_snapshot_id", source["snapshot_id"]),
                    "portable_origin": metadata.get("portable_origin"),
                    "raw_locator": "Drive file ID -> optional inner_tar -> suffix after # in document.raw_file",
                    "raw_verification": "METADATA_ONLY_RAW_PAYLOAD_NOT_REPLAYED"})
            entry = selection["files"]["selection"]
            verify_file(target/"selection"/entry["path"], entry["sha256"])
            counts = intake_core._write_outcomes(target, target/"selection", selection, build_dir, build, self.work)
            with _connection(self.work) as con:
                if build:
                    con.read_parquet([str(build_dir/p["files"]["documents"]["path"]) for p in build["parts"]]).create_view("added")
                    query = "SELECT doc_id,url FROM added"
                else:
                    query = "SELECT NULL::BIGINT AS doc_id,NULL::VARCHAR AS url WHERE false"
                intake_core._parquet(con, query, target/"added_ids.parquet", self.work)
            files = {kind: {"path": name, "sha256": sha256_file(target/name), "bytes": (target/name).stat().st_size}
                     for kind, name in (("added_ids", "added_ids.parquet"), ("outcomes", "source_outcomes.parquet"))}
            saved = {"source": source, "plan_sha256": self.plan_sha, "state": "COMPLETE", "counts": counts,
                "build_run": job if build else None, "build_snapshot_sha256": build["snapshot_sha256"] if build else None,
                "data_counts": build["counts"] if build else {}, "files": files,
                "added_ids_sha256": files["added_ids"]["sha256"], "outcomes_sha256": files["outcomes"]["sha256"],
                "updated_at": utc_now()}
            atomic_json(target/"complete.json", saved)  # last: durable commit of the whole source
            self.completed.append(saved)
            existing.append((target/"added_ids.parquet", saved["added_ids_sha256"]))
            result = self.status()
            atomic_json(self.root/"intake_status.json", result)
            print(f"SOURCE_CHECKPOINT_SAVED {job} | {counts}", flush=True)
            if source_completed is not None:
                source_completed()
            newly_completed += 1
            if max_new_sources is not None and newly_completed >= max_new_sources:
                return result
        return result

"""Audit/import the later 00008 archive without changing the completed campaign."""
from copy import deepcopy
import hashlib
import json
from pathlib import Path
import re
import tarfile

from vietmedbridge.artifacts import atomic_json, digest_json, read_json
from vietmedbridge.external_import import MEMBERS
from vietmedbridge.isolated_intake import _inside

SOURCE_00008 = {'drive_file_id':'1lqL46iCpfD7qnub-8ih1avKj6K3Dporv',
    'archive':'vibiomir_retain_shard_00008.tar','archive_bytes':2503065600,
    'inner_tar':None,'job_id':'new-00008','mode':'NEW_SNAPSHOT_SOURCE',
    'snapshot_id':'5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6',
    'frontier':'data/crawl_shards/shard_00008.parquet'}


def _checked(value):
    if value.get('manifest_sha256')!=digest_json({k:v for k,v in value.items() if k!='manifest_sha256'}):
        raise ValueError('Extension audit/manifest digest changed.')
    return value


def completed_parent_mappings(parent,scope):
    """Read nine commit receipts and file sizes; payload SHA is deferred to use."""
    parent=Path(parent)
    report=read_json(parent/'full_intake_status.json')
    groups=[('.',scope['base_plan']['sources']),('additional-frontiers',scope['additional_sources']),
        ('retained-recovery',[scope['retained']])]
    expected={s['job_id'] for _,sources in groups for s in sources}
    if (report.get('state')!='FULL_INTAKE_COMPLETE' or report.get('completed_sources')!=len(expected)
        or report.get('total_sources')!=len(expected) or report.get('pending_sources')
        or report.get('scope_sha256')!=digest_json(scope)
        or {b['build_run'] for b in report.get('builds',[])}!=expected):
        raise ValueError('The nine-source parent campaign is incomplete or differs from the audited scope.')
    refs,counts,seen=[],{},set()
    for phase,sources in groups:
        root=_inside(parent,phase)
        plan=read_json(root/'intake_plan.json')
        pinned={s['job_id']:s for s in plan['sources']}
        if set(pinned)!={s['job_id'] for s in sources}:
            raise ValueError('Parent phase sources changed.')
        for source in sources:
            job=source['job_id'];target=_inside(root,'sources/'+job)
            saved=read_json(target/'complete.json')
            if (saved.get('state')!='COMPLETE' or saved.get('source')!=pinned[job]
                or saved.get('plan_sha256')!=digest_json(plan)
                or any(saved['source'].get(k)!=source[k] for k in ('drive_file_id','snapshot_id','job_id'))
                or saved.get('build_run')!=job or job in seen):
                raise ValueError('Parent source receipt changed; no automatic replay.')
            item=saved['files']['added_ids']
            path=_inside(target,item['path'])
            if (item['sha256']!=saved['added_ids_sha256'] or path.stat().st_size!=item['bytes']
                or not re.fullmatch('[0-9a-f]{64}',item['sha256'])):
                raise ValueError('Parent mapping receipt/file size changed.')
            refs.append((path,item['sha256']));seen.add(job)
            for name,value in saved['counts'].items():
                counts[name]=counts.get(name,0)+value
    if counts!=report['counts'] or sum(counts.values())!=scope['audited_coverage']['union_native_ids']:
        raise ValueError('Parent committed source counts do not match the campaign.')
    return refs,report


def audit_00008(root,load_reader,*,source=SOURCE_00008):
    """Seek TAR headers, read descriptor/frontier only; never copy the full TAR."""
    root=Path(root);marker=root/'source_audit.json'
    if marker.is_file():
        saved=_checked(read_json(marker))
        frontier=saved.get('metadata',{}).get('frontier',{}).get('sha256','')
        if (saved.get('request')!=source or not re.fullmatch('[0-9a-f]{64}',frontier)
            or saved.get('source')!=dict(source,frontier_sha256=frontier)):
            raise ValueError('Extension source changed; retain this campaign and create a new one.')
        return saved['source']
    reader=load_reader();remote=reader.metadata(source['drive_file_id'])
    if remote.get('name')!=source['archive'] or int(remote['size'])!=source['archive_bytes']:
        raise ValueError('00008 archive identity/size changed; re-audit before import.')
    wanted={MEMBERS['dataset']:'dataset',source['frontier']:'frontier'}
    found={}
    with reader.open(source['drive_file_id'],remote) as stream, tarfile.open(fileobj=stream,mode='r:') as archive:
        for index,member in enumerate(archive):
            if index>=128:
                raise ValueError('00008 required metadata missing from bounded TAR header scan.')
            kind=wanted.get(member.name)
            if kind is None:
                continue
            if kind in found or not member.isfile() or not 0<member.size<=512*2**20:
                raise ValueError('Invalid/duplicate 00008 metadata member.')
            if kind=='dataset' and member.size>2**20:
                raise ValueError('Dataset descriptor exceeds its bounded JSON size.')
            sha=hashlib.sha256();pieces=[];received=0
            with archive.extractfile(member) as incoming:
                for block in iter(lambda:incoming.read(2**20),b''):
                    sha.update(block)
                    received+=len(block)
                    if kind=='dataset':
                        pieces.append(block)
            checksum=sha.hexdigest()
            if received!=member.size:
                raise EOFError('Truncated 00008 metadata member.')
            expected=member.pax_headers.get('vibiomir.sha256')
            if expected is not None and checksum!=expected:
                raise ValueError('00008 member SHA differs from its TAR metadata.')
            found[kind]={'sha256':checksum,'bytes':member.size,'path':member.name}
            if kind=='dataset':
                descriptor=json.loads(b''.join(pieces))
                if descriptor.get('snapshot_id')!=source['snapshot_id']:
                    raise ValueError('00008 belongs to a different BTC snapshot.')
            if len(found)==2:
                break
    if set(found)!={'dataset','frontier'}:
        raise ValueError('00008 descriptor/frontier not found; do not guess its source layout.')
    if reader.metadata(source['drive_file_id'])!=remote:
        raise ValueError('00008 archive changed during audit.')
    bound=dict(source,frontier_sha256=found['frontier']['sha256'])
    saved={'request':source,'source':bound,'remote':remote,'metadata':found,
        'scope':'Descriptor/frontier identity only; row membership and all extraction bindings checked during intake'}
    saved['manifest_sha256']=digest_json(saved)
    atomic_json(marker,saved)
    print('00008 SOURCE_AUDITED: snapshot/frontier bound; full archive not downloaded',flush=True)
    return bound


def load_00008_metadata(reader,source,target,audit_root):
    from vietmedbridge.isolated_intake import stage_metadata
    audit=_checked(read_json(Path(audit_root)/'source_audit.json'))
    if source!=audit['source'] or reader.metadata(source['drive_file_id'])!=audit['remote']:
        raise ValueError('00008 archive version changed after audit; do not change the saved campaign.')
    return stage_metadata(reader,source,target)


def extension_00008_plan(parent,root,work,scope,load_reader,*,source=SOURCE_00008):
    parent,root,work=Path(parent).resolve(),Path(root).resolve(),Path(work).resolve()
    if (root==parent or root.is_relative_to(parent) or parent.is_relative_to(root)
        or work.is_relative_to(parent) or parent.is_relative_to(work)
        or work.is_relative_to(root) or root.is_relative_to(work)):
        raise ValueError('00008 must use a separate sibling campaign and runtime work directory.')
    refs,report=completed_parent_mappings(parent,scope)
    if (source['snapshot_id']!=scope['base_plan']['required_snapshot_id']
        or source['snapshot_id'] not in scope['base_plan']['snapshots']):
        raise ValueError('Extra source differs from the parent BTC snapshot.')
    source=audit_00008(root,load_reader,source=source)
    if source['frontier_sha256'] in {s['frontier_sha256'] for s in scope['base_plan']['sources']+scope['additional_sources']}:
        raise ValueError('00008 repeats an already-committed frontier.')
    plan=deepcopy(scope['base_plan'])
    plan.update(sources=[source],campaign='btc-new-71be0e12-00008-20261020-v1',
        parent_scope_sha256=digest_json(scope),prior_mapping_sha256=[checksum for _,checksum in refs])
    return plan,refs,report


from vietmedbridge.artifacts import verify_file, publish_file
from vietmedbridge.isolated_intake import DriveSource, fetch_reference
from vietmedbridge.chunks import load_bge_tokenizer, ChunkConfig
from vietmedbridge.golden import run_golden_suite
from huggingface_hub import hf_hub_download
RUN_MODE = "sequential"
REFERENCE_DIR = INTAKE_ROOT / "references"

LAZY_RESOURCES = {}

def load_reader():
    if "reader" not in LAZY_RESOURCES:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
        auth.authenticate_user()
        credentials, _ = google.auth.default()
        LAZY_RESOURCES["reader"] = DriveSource(AuthorizedSession(credentials))
    return LAZY_RESOURCES["reader"]

def load_baseline():
    if "baseline" not in LAZY_RESOURCES:
        reference = PLAN["baseline"]
        shared = PARENT_ROOT / "references/baseline_aliases.parquet"
        target = shared if shared.is_file() else REFERENCE_DIR / "baseline_aliases.parquet"
        if not target.is_file():
            fetch_reference(load_reader(), reference, target)
        print("Cần mapping cũ để chọn phần mới; chỉ đưa mapping lên ổ local một lần.", flush=True)
        LAZY_RESOURCES["baseline"] = intake_core._local_mapping(target, reference["sha256"], WORK_DIR)
    return LAZY_RESOURCES["baseline"]

def load_links(snapshot_id):
    key = "links-"+snapshot_id
    if key not in LAZY_RESOURCES:
        spec = PLAN["snapshots"][snapshot_id]
        shared = PARENT_ROOT / "references" / (snapshot_id[:16]+"-links.parquet")
        target = shared if shared.is_file() else REFERENCE_DIR / (snapshot_id[:16]+"-links.parquet")
        if not target.is_file():
            print("Tải official links:", spec["revision"], flush=True)
            downloaded = Path(hf_hub_download("AIGuruTinix/ViBioMIR", spec["filename"],
                repo_type="dataset", revision=spec["revision"]))
            verify_file(downloaded, spec["sha256"])
            publish_file(downloaded, target)
        LAZY_RESOURCES[key] = intake_core._local_mapping(target, spec["sha256"], WORK_DIR)
    return LAZY_RESOURCES[key]

def load_tokenizer():
    if "tokenizer" not in LAZY_RESOURCES:
        tokenizer, spec = load_bge_tokenizer(PLAN["tokenizer_revision"])
        golden = run_golden_suite(CHECKOUT / "tests/golden/cases.json", tokenizer, spec, CONFIG)
        atomic_json((INTAKE_ROOT if RUN_MODE == "sequential" else REFERENCE_DIR.parent) / "golden.json", golden)
        if not golden["passed"]:
            raise RuntimeError("Golden failed; chưa chia chunk mới.")
        LAZY_RESOURCES["tokenizer"] = (tokenizer, spec)
    return LAZY_RESOURCES["tokenizer"]

FULL_SCOPE = json.loads('{"schema_version": 1, "audit_date": "2026-10-19", "folder_id": "1P2ke-yIDwj443R_GMegaiH9RACf-Gzth", "base_plan_sha256": "dd4d09120f1e8c0c9c3dc1d977ce34c73e5ac97a58d2df6d56fdc2f13f6fae3f", "base_plan": {"state": "AUDITED_NOT_IMPORTED", "audit_date": "2026-10-19", "active_candidate_action": "KEEP_UNCHANGED", "new_snapshot": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "sources": [{"drive_file_id": "1tVX24iYCthJSYyMsKLdoVBJrgWtUmG-k", "archive": "vibiomir_retain_shard_00001_with_resume_00000.tar", "inner_tar": "vibiomir_shard_00000.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00000.parquet", "frontier_sha256": "2cf6be6537b6e4c73b25c421503a9e5b7247dddbd10abe8749ae0ac1dbcefcec", "extract_success_url_count_before_pipeline": 11620, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00000", "archive_bytes": 5262612480}, {"drive_file_id": "1IwebnPLyQhXTphcRtfI7Pg1aZEiXdBEZ", "archive": "vibiomir_retain_shard_00002_with_resume_00001.tar", "inner_tar": "vibiomir_shard_00001.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00001.parquet", "frontier_sha256": "7b5669ff5fe407205075957696fe846471c2c16cd3a91509406db6caf884c3f0", "extract_success_url_count_before_pipeline": 69498, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00001", "archive_bytes": 7523225600}, {"drive_file_id": "1IwebnPLyQhXTphcRtfI7Pg1aZEiXdBEZ", "archive": "vibiomir_retain_shard_00002_with_resume_00001.tar", "inner_tar": "vibiomir_shard_00002.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00002.parquet", "frontier_sha256": "36b93a5a17933403be8de662dd8a985eeb7287025201e056b61970b05f8b7a52", "extract_success_url_count_before_pipeline": 58976, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00002", "archive_bytes": 7523225600}, {"drive_file_id": "1Z8_0BlP-WwVYkfARLDR7vUtRCa7Xh-69", "archive": "vibiomir_retain_shard_00004_with_resume_00003.tar", "inner_tar": "vibiomir_shard_00003.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00003.parquet", "frontier_sha256": "cfc373b543d0253978a5fcd27c5f845ce64fbcd8ef349a137ee838ec51d3c2b9", "extract_success_url_count_before_pipeline": 69237, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00003", "archive_bytes": 7277260800}, {"drive_file_id": "1Z8_0BlP-WwVYkfARLDR7vUtRCa7Xh-69", "archive": "vibiomir_retain_shard_00004_with_resume_00003.tar", "inner_tar": "vibiomir_shard_00004.tar", "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00004.parquet", "frontier_sha256": "18a8db71eefd5b63874e229d42c6411dadd5efbdf810ed01bdc4e977295c2f32", "extract_success_url_count_before_pipeline": 56845, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00004", "archive_bytes": 7277260800}, {"drive_file_id": "11Yl5P1lFYaCZbcydc5vsUPK2LyJ5fkyC", "archive": "vibiomir_retain_shard_00005.tar", "inner_tar": null, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00005.parquet", "frontier_sha256": "c102e6c47d672ff031d8eeb227d5491a0f49409bed8b1a24443bed2ea94aa1aa", "extract_success_url_count_before_pipeline": 55323, "mode": "NEW_SNAPSHOT_SOURCE", "job_id": "new-00005", "archive_bytes": 3269468160}], "rules": ["Open nested TAR members explicitly; do not import outer containers as flat archives", "Use exactly one cumulative version of each canonical frontier; preserve original archives", "Bind new shards to their new links snapshot; never relabel them as old-snapshot inputs", "Deduplicate against existing valid documents by exact URL/ID; preserve failed/pending URL ledger entries", "Keep ongoing submission candidate frozen; stage additions without switching active pointers", "EXTRACT_SUCCESS goes through normal quality/chunk gates; only new text hashes need embedding", "Only import sources from the new BTC snapshot; exclude all legacy-snapshot archive jobs"], "schema_version": 1, "campaign": "btc-new-71be0e12-20261019-v1", "tokenizer_revision": "5617a9f61b028005a4858fdac845db406aefb181", "chunking": {"child_tokens": 180, "overlap_tokens": 40, "parent_tokens": 512, "snap_sentences": true, "prefer_sections": true, "dense_max_tokens": 512}, "baseline": {"drive_file_id": "1JBnjci3WUfBlQ0VxKtGeGHTbmhGpydRp", "sha256": "e34ea4e336ea37e42ffb338a3a97986c816b932fdc221c1843d5d3424782ab0c", "candidate_manifest_sha256": "4de4b5c77b867e6017b0f93b56e6d32e1e0c9442ab05d4bdb8e14673e4f7bcda", "documents": 1252445}, "snapshots": {"5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6": {"revision": "71be0e1239a19dfd7a717e2bc13e3bec21bd0334", "filename": "links_corpus_vi_en_zh.parquet", "sha256": "87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7"}}, "required_snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "scope": "NEW_BTC_DATASET_ONLY"}, "additional_sources": [{"drive_file_id": "18cg2bXqfX-6f867we7tdcUEseG52HbZY", "archive": "vibiomir_retain__shard_00006.tar", "archive_bytes": 3489576960, "inner_tar": null, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00006.parquet", "frontier_sha256": "06823daba41b1be5b6129f961f4a082e14e12e50a4a42e1d6d7335a5f1165296", "extract_success_url_count_before_pipeline": 63806, "job_id": "new-00006", "mode": "NEW_SNAPSHOT_SOURCE"}, {"drive_file_id": "1uA45mDiRXs2Qlb-b0jPqHU9GavA7hHKh", "archive": "vibiomir_retain_shard_00007.tar", "archive_bytes": 3993866240, "inner_tar": null, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "frontier": "data/crawl_shards/shard_00007.parquet", "frontier_sha256": "e9642c5ab759b21ea320793409c8675809af4d8d171562a6ffe9acfcde444800", "extract_success_url_count_before_pipeline": 69940, "job_id": "new-00007", "mode": "NEW_SNAPSHOT_SOURCE"}], "retained": {"job_id": "retained-71be0e12", "mode": "NEW_SNAPSHOT_SOURCE", "format": "RETAINED_PORTABLE_V1", "archive": "vibiomir_retained_71be0e12.tar", "drive_file_id": "1Hf6cb7Des8AhYkyWXE3d3PvnDT-BMMQ-", "archive_bytes": 12581928960, "snapshot_id": "5586f423916005c11ffa8afb96895017d672af51d68aee8a88d28dd2e0dc39c6", "original_snapshot_id": "b915d5d5d640d497d103fa37a5a047c1d093772ab17b09638634de4bbc80e5d7", "official_links_sha256": "87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7", "members": {"mapping": {"path": "data/doc_url_map.parquet", "offset": 12562358272, "bytes": 19545473, "sha256": "1dc06041fded26cfe2dfa0c64e39875840a8b3a8051ac58080608497fadaee01"}, "raw_catalog": {"path": "data/raw_catalog.parquet", "offset": 11313683456, "bytes": 125719140, "sha256": "90e5ae894488dd7e47a76dd4664bcaa8156df72245f18c078db1b66ac0f4f9e9"}, "documents": {"path": "data/documents.parquet", "offset": 11439404544, "bytes": 1122951853, "sha256": "b031ec37043f1de3e6f24a70e508fd3e569fb8dff319a59c69489a3bd8adabc3"}, "dataset": {"path": "new_dataset_manifest.json", "offset": 12581909504, "bytes": 1093, "sha256": "912ac3c910985b5f3f003f3d90ea317b13c9c568da7baf424d87488b1cfe7d31"}}}, "required_physical_archive_names": ["vibiomir_retain_shard_00001_with_resume_00000.tar", "vibiomir_retain_shard_00002_with_resume_00001.tar", "vibiomir_retain_shard_00004_with_resume_00003.tar", "vibiomir_retain_shard_00005.tar", "vibiomir_retain__shard_00006.tar", "vibiomir_retain_shard_00007.tar", "vibiomir_retained_71be0e12.tar"], "duplicate_versions_to_skip": ["vibiomir_retain_shard_00000.tar", "vibiomir_retain_shard_00003.tar", "inner vibiomir_shard_00001.tar in vibiomir_retain_shard_00001_with_resume_00000.tar"], "audited_coverage": {"frontier_urls": 800000, "frontier_native_ids": 800999, "retained_urls": 316097, "retained_native_ids": 316693, "union_native_ids": 1117692, "official_native_ids": 1220598, "note": "Input membership coverage, not successful extraction or completed pipeline coverage"}}')

PLAN, PRIOR_MAPPINGS, PARENT_STATUS = extension_00008_plan(
    PARENT_ROOT, INTAKE_ROOT, WORK_DIR, FULL_SCOPE, load_reader)
CONFIG = ChunkConfig(**PLAN["chunking"])
TOKENIZER_SPEC = {"model_id":"BAAI/bge-m3", "revision":PLAN["tokenizer_revision"], "special_tokens":False}
CONTROLLER = IntakeResume(PLAN, root=INTAKE_ROOT, work_dir=WORK_DIR,
    tokenizer_spec=TOKENIZER_SPEC, chunk_config=CONFIG, shard_size=2048,
    prior_mappings=PRIOR_MAPPINGS)
print("Parent giữ nguyên:", PARENT_STATUS["completed_sources"], "/", PARENT_STATUS["total_sources"],flush=True)
print(json.dumps(CONTROLLER.status(),ensure_ascii=False,indent=2),flush=True)


In [ ]:

#@title 3. Nhập riêng 00008: bỏ qua ID đã hợp lệ, chia chunk phần mới, giữ lỗi để xử lý sau
def source_metadata(source,target):
    return load_00008_metadata(load_reader(),source,target,INTAKE_ROOT)
def continue_00008():
    return CONTROLLER.run(load_reader=load_reader,load_links=load_links,
        load_baseline=load_baseline,load_tokenizer=load_tokenizer,load_metadata=source_metadata)
try:
    RESULT = continue_00008()
except OSError as error:
    if error.errno not in (103,107):
        raise
    print("Drive ngắt kết nối; giữ checkpoint và gắn lại một lần.",flush=True)
    drive.mount("/content/drive",force_remount=True)
    RESULT = continue_00008()
atomic_json(INTAKE_ROOT / "intake_status.json",RESULT)
print(json.dumps(RESULT,ensure_ascii=False,indent=2),flush=True)
if RESULT["state"] == "INTAKE_COMPLETE":
    print("00008 CPU COMPLETE. Báo cáo:", INTAKE_ROOT / "intake_status.json",flush=True)
    print("Chưa embedding/freeze. Parent 9 nguồn và candidate submit hiện tại giữ nguyên.",flush=True)
